# Price Segmentation Test V1

I created this notebook inside `ml/test` to test the segmented or mixture-of-experts pricing idea. The goal is to see whether splitting normal listings and luxury listings can improve the current single-model nightly price benchmark.


## Setup Note

In this cell I import the shared ML helper, the scoring utilities, and the classifier I need for the segmentation routing experiment.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

BASE_DIR = Path.cwd()
SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR / 'code files' / 'ml',
    BASE_DIR / 'code files' / 'ml' / 'test',
    BASE_DIR.parent,
    BASE_DIR.parent / 'ml',
    BASE_DIR.parent.parent,
    BASE_DIR.parent.parent / 'ml',
]
for candidate in SEARCH_DIRS:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from gold_ml_modeling import (
    build_mysql_engine,
    load_property_mart_frame,
    prepare_common_features,
    add_train_only_price_proxies,
    build_candidate_models,
    build_pipeline,
    PRICE_CATEGORICAL_COLUMNS,
    PRICE_LAUNCH_CORE_FEATURES,
    PRICE_MARKET_PROXY_FEATURES,
)

TEST_DIR = Path.cwd()
TEST_DIR


## Data Load Note

In this cell I load the current Gold property mart, prepare the shared features, and keep only rows that have a nightly price target available.


In [ ]:
engine = build_mysql_engine()
model_df = load_property_mart_frame(engine)
prepared_df = prepare_common_features(model_df).dropna(subset=['target_nightly_price']).copy()
prepared_df.shape


## Feature Design Note

In this cell I define the pricing feature families I want to use in the experiment. I keep the same strong market-anchor feature design from the active pricing notebook so the comparison stays fair.


In [ ]:
quality_columns = [
    'host_is_superhost','platform_count','number_of_reviews','reviews_per_month','review_scores_rating','review_scores_cleanliness','review_scores_location','review_scores_value',
    'host_response_rate','host_acceptance_rate','host_listings_count','host_total_listings_count','host_tenure_days','host_tenure_years','review_score_blend'
]
availability_columns = ['availability_30','availability_60','availability_90','availability_365']
financing_columns = ['financing_vintage_year','applied_asset_discount_pct','applied_annual_interest_rate']
feature_sets = {
    'launch_plus_market_quality': PRICE_LAUNCH_CORE_FEATURES + PRICE_MARKET_PROXY_FEATURES + quality_columns,
    'observed_market_full': PRICE_LAUNCH_CORE_FEATURES + PRICE_MARKET_PROXY_FEATURES + quality_columns + availability_columns + financing_columns,
}
feature_sets.keys()


## Split Note

In this cell I split the data into train, validation, and test. I use validation data to choose the segmentation threshold and the segmented model configuration, and only after that do I evaluate the chosen version on the test split.


In [ ]:
X = prepared_df.drop(columns=['target_nightly_price'])
y = prepared_df['target_nightly_price']
X_train_all, X_test, y_train_all, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_inner, X_valid, y_train_inner, y_valid = train_test_split(X_train_all, y_train_all, test_size=0.2, random_state=42)
X_train_inner, X_valid = add_train_only_price_proxies(X_train_inner, X_valid, y_train_inner)
len(X_train_inner), len(X_valid), len(X_test)


## Baseline Validation Note

In this cell I benchmark the single-model baselines on the validation split so I have a fair reference before testing the segmented routing idea.


In [ ]:
models = build_candidate_models(include_linear=True, include_dummy=False)
regressors = {k: v for k, v in models.items() if k in {'xgboost', 'hist_gradient_boosting', 'ridge'}}

baseline_rows = []
for fs_name, cols in feature_sets.items():
    for target_mode in ['raw_price', 'log_price']:
        y_fit = np.log1p(y_train_inner) if target_mode == 'log_price' else y_train_inner
        for name, model in regressors.items():
            pipe = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(model))
            pipe.fit(X_train_inner[cols], y_fit)
            pred = pipe.predict(X_valid[cols])
            if target_mode == 'log_price':
                pred = np.expm1(pred)
            pred = np.clip(pred, 0, None)
            baseline_rows.append({
                'feature_set': fs_name,
                'target_mode': target_mode,
                'model_name': name,
                'mae': mean_absolute_error(y_valid, pred),
                'rmse': np.sqrt(mean_squared_error(y_valid, pred)),
                'r2': r2_score(y_valid, pred),
            })
baseline_validation_df = pd.DataFrame(baseline_rows).sort_values(['rmse', 'mae']).reset_index(drop=True)
baseline_validation_df.head(10)


## Segmented Validation Note

In this cell I test the segmented pricing idea. I tune the luxury threshold using the training target only, train a classifier to route normal versus luxury listings, and compare hard versus soft routing on the validation split.


In [ ]:
segmented_rows = []
classifier_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    eval_metric='logloss',
    n_jobs=1,
)

for fs_name, cols in feature_sets.items():
    Xtr = X_train_inner[cols]
    Xva = X_valid[cols]
    for q in [0.90, 0.925, 0.95, 0.975]:
        threshold = y_train_inner.quantile(q)
        luxury_flag_train = (y_train_inner >= threshold).astype(int)
        luxury_mask_arr = luxury_flag_train.to_numpy().astype(bool)
        if luxury_mask_arr.sum() < 50:
            continue

        clf = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(classifier_model))
        clf.fit(Xtr, luxury_flag_train)
        prob = clf.predict_proba(Xva)[:, 1]
        hard_pred_class = (prob >= 0.5).astype(int)

        valid_is_lux = (y_valid >= threshold).to_numpy()
        valid_is_norm = ~valid_is_lux

        for target_mode in ['raw_price', 'log_price']:
            for normal_name, normal_model in regressors.items():
                for luxury_name, luxury_model in regressors.items():
                    y_fit_normal = np.log1p(y_train_inner[~luxury_mask_arr]) if target_mode == 'log_price' else y_train_inner[~luxury_mask_arr]
                    y_fit_lux = np.log1p(y_train_inner[luxury_mask_arr]) if target_mode == 'log_price' else y_train_inner[luxury_mask_arr]

                    normal_pipe = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(normal_model))
                    luxury_pipe = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(luxury_model))
                    normal_pipe.fit(Xtr.iloc[~luxury_mask_arr], y_fit_normal)
                    luxury_pipe.fit(Xtr.iloc[luxury_mask_arr], y_fit_lux)

                    pred_normal = normal_pipe.predict(Xva)
                    pred_lux = luxury_pipe.predict(Xva)
                    if target_mode == 'log_price':
                        pred_normal = np.expm1(pred_normal)
                        pred_lux = np.expm1(pred_lux)
                    pred_normal = np.clip(pred_normal, 0, None)
                    pred_lux = np.clip(pred_lux, 0, None)

                    hard_pred = np.where(hard_pred_class == 1, pred_lux, pred_normal)
                    soft_pred = prob * pred_lux + (1 - prob) * pred_normal
                    yv = y_valid.to_numpy()

                    for route_name, pred in [('hard_route', hard_pred), ('soft_route', soft_pred)]:
                        segmented_rows.append({
                            'feature_set': fs_name,
                            'threshold_q': q,
                            'threshold_value': float(threshold),
                            'target_mode': target_mode,
                            'normal_model': normal_name,
                            'luxury_model': luxury_name,
                            'route': route_name,
                            'mae': mean_absolute_error(yv, pred),
                            'rmse': np.sqrt(mean_squared_error(yv, pred)),
                            'r2': r2_score(yv, pred),
                            'mae_normal_only': mean_absolute_error(yv[valid_is_norm], pred[valid_is_norm]),
                            'mae_luxury_only': mean_absolute_error(yv[valid_is_lux], pred[valid_is_lux]),
                            'rmse_normal_only': np.sqrt(mean_squared_error(yv[valid_is_norm], pred[valid_is_norm])),
                            'rmse_luxury_only': np.sqrt(mean_squared_error(yv[valid_is_lux], pred[valid_is_lux])),
                            'luxury_train_rows': int(luxury_mask_arr.sum()),
                        })

segmented_validation_df = pd.DataFrame(segmented_rows).sort_values(['rmse', 'mae']).reset_index(drop=True)
segmented_validation_df.to_csv(TEST_DIR / 'price_segmentation_validation_results_v1.csv', index=False)
segmented_validation_df.head(15)


## Test Comparison Note

In this cell I refit the best segmented validation configuration on the full training split and compare it against the current single-model pricing winner on the held-out test split.


In [ ]:
best_cfg = segmented_validation_df.iloc[0].to_dict()

X_train_full, X_test_full = add_train_only_price_proxies(X_train_all, X_test, y_train_all)
q = float(best_cfg['threshold_q'])
threshold = y_train_all.quantile(q)
luxury_flag_full = (y_train_all >= threshold).astype(int)
luxury_mask_full = luxury_flag_full.to_numpy().astype(bool)
cols = feature_sets[best_cfg['feature_set']]

clf = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(classifier_model))
clf.fit(X_train_full[cols], luxury_flag_full)
prob = clf.predict_proba(X_test_full[cols])[:, 1]
hard_pred_class = (prob >= 0.5).astype(int)

normal_pipe = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(regressors[best_cfg['normal_model']]))
luxury_pipe = build_pipeline(cols, PRICE_CATEGORICAL_COLUMNS, clone(regressors[best_cfg['luxury_model']]))
if best_cfg['target_mode'] == 'log_price':
    normal_pipe.fit(X_train_full.iloc[~luxury_mask_full][cols], np.log1p(y_train_all.iloc[~luxury_mask_full]))
    luxury_pipe.fit(X_train_full.iloc[luxury_mask_full][cols], np.log1p(y_train_all.iloc[luxury_mask_full]))
else:
    normal_pipe.fit(X_train_full.iloc[~luxury_mask_full][cols], y_train_all.iloc[~luxury_mask_full])
    luxury_pipe.fit(X_train_full.iloc[luxury_mask_full][cols], y_train_all.iloc[luxury_mask_full])

pred_normal = normal_pipe.predict(X_test_full[cols])
pred_lux = luxury_pipe.predict(X_test_full[cols])
if best_cfg['target_mode'] == 'log_price':
    pred_normal = np.expm1(pred_normal)
    pred_lux = np.expm1(pred_lux)
pred_normal = np.clip(pred_normal, 0, None)
pred_lux = np.clip(pred_lux, 0, None)
segmented_pred = np.where(hard_pred_class == 1, pred_lux, pred_normal) if best_cfg['route'] == 'hard_route' else (prob * pred_lux + (1 - prob) * pred_normal)

baseline_cols = feature_sets['observed_market_full']
baseline_pipe = build_pipeline(baseline_cols, PRICE_CATEGORICAL_COLUMNS, clone(regressors['hist_gradient_boosting']))
baseline_pipe.fit(X_train_full[baseline_cols], np.log1p(y_train_all))
baseline_pred = np.expm1(baseline_pipe.predict(X_test_full[baseline_cols]))
baseline_pred = np.clip(baseline_pred, 0, None)

yt = y_test.to_numpy()
lux_mask_test = (y_test >= threshold).to_numpy()
norm_mask_test = ~lux_mask_test

comparison_df = pd.DataFrame([
    {
        'model_variant': 'baseline_single_hgb_log_observed_full',
        'mae': mean_absolute_error(yt, baseline_pred),
        'rmse': np.sqrt(mean_squared_error(yt, baseline_pred)),
        'r2': r2_score(yt, baseline_pred),
        'mae_normal_only': mean_absolute_error(yt[norm_mask_test], baseline_pred[norm_mask_test]),
        'mae_luxury_only': mean_absolute_error(yt[lux_mask_test], baseline_pred[lux_mask_test]),
        'rmse_normal_only': np.sqrt(mean_squared_error(yt[norm_mask_test], baseline_pred[norm_mask_test])),
        'rmse_luxury_only': np.sqrt(mean_squared_error(yt[lux_mask_test], baseline_pred[lux_mask_test])),
    },
    {
        'model_variant': 'segmented_best_validation',
        'mae': mean_absolute_error(yt, segmented_pred),
        'rmse': np.sqrt(mean_squared_error(yt, segmented_pred)),
        'r2': r2_score(yt, segmented_pred),
        'mae_normal_only': mean_absolute_error(yt[norm_mask_test], segmented_pred[norm_mask_test]),
        'mae_luxury_only': mean_absolute_error(yt[lux_mask_test], segmented_pred[lux_mask_test]),
        'rmse_normal_only': np.sqrt(mean_squared_error(yt[norm_mask_test], segmented_pred[norm_mask_test])),
        'rmse_luxury_only': np.sqrt(mean_squared_error(yt[lux_mask_test], segmented_pred[lux_mask_test])),
    },
])
comparison_df.to_csv(TEST_DIR / 'price_segmentation_test_summary_v1.csv', index=False)
best_cfg, comparison_df


## Interpretation Note

This final cell keeps the decision clear. I can now see whether the segmented pricing idea improved the overall score and whether it reduced the luxury-segment errors without hiding the trade-off on normal listings.


In [ ]:
decision = pd.Series({
    'current_single_model_r2': float(comparison_df.loc[comparison_df['model_variant'] == 'baseline_single_hgb_log_observed_full', 'r2'].iloc[0]),
    'segmented_model_r2': float(comparison_df.loc[comparison_df['model_variant'] == 'segmented_best_validation', 'r2'].iloc[0]),
    'current_single_model_rmse': float(comparison_df.loc[comparison_df['model_variant'] == 'baseline_single_hgb_log_observed_full', 'rmse'].iloc[0]),
    'segmented_model_rmse': float(comparison_df.loc[comparison_df['model_variant'] == 'segmented_best_validation', 'rmse'].iloc[0]),
    'current_single_model_mae': float(comparison_df.loc[comparison_df['model_variant'] == 'baseline_single_hgb_log_observed_full', 'mae'].iloc[0]),
    'segmented_model_mae': float(comparison_df.loc[comparison_df['model_variant'] == 'segmented_best_validation', 'mae'].iloc[0]),
    'recommended_winner_by_score': 'segmented_best_validation' if comparison_df.loc[comparison_df['model_variant'] == 'segmented_best_validation', 'r2'].iloc[0] > comparison_df.loc[comparison_df['model_variant'] == 'baseline_single_hgb_log_observed_full', 'r2'].iloc[0] else 'baseline_single_hgb_log_observed_full',
})
decision
